# ENAHO — Exploración de la estructura de microdatos

**Issue #13** · `area:datos` · Milestone M3

Objetivo: entender la estructura de los microdatos de la **ENAHO** (Encuesta Nacional de Hogares, INEI) para poder, en las issues siguientes, derivar la canasta básica de consumo por departamento.

- **#13 (esta)** — explorar estructura: módulos, columnas, códigos de producto, factor de expansión.
- **#17** — diseñar la tabla `canasta_consumo_dept`.
- **#19** — construir la canasta básica representativa por departamento.

Productos objetivo del MVP: **papa, limón, pollo, cebolla, huevo, tomate**.

---

## Cómo obtener los datos

La descarga es **manual y one-shot** (el portal del INEI geo-bloquea IPs de datacenter y los microdatos pesan cientos de MB):

1. Ir a `https://www.inei.gob.pe/microdatos` → *Consulta por Encuestas* → **ENAHO Metodología ACTUALIZADA** → año más reciente disponible (Condiciones de Vida y Pobreza).
2. Descargar al menos: **Módulo 601 — Gastos en Alimentos y Bebidas** y **Sumaria** (formato SPSS / `.sav`).
3. Descomprimir los `.sav` en `data/enaho/<ANIO>/` (carpeta **ignorada por git** — nunca se versiona).
4. (Recomendado) Subir los crudos a R2 con la última celda de este notebook, para que sean reproducibles por el resto del equipo sin re-descargar.

Fuente de verdad: **R2** (`enaho/<ANIO>/`, inmutable). Copia local en `data/enaho/<ANIO>/` solo para velocidad.

## 0. Setup y parámetros

El año está **parametrizado**: cuando el INEI publique un año más reciente, basta cambiar `ANIO`.

In [ ]:
from pathlib import Path

import pandas as pd
import pyreadstat
from dotenv import load_dotenv

load_dotenv()  # credenciales R2 desde el .env de la raíz del repo

# Año de la ENAHO a explorar. Regla del equipo: usar 2025 si el INEI ya la
# publicó; si todavía no está disponible, caer a 2024.
ANIO = 2024

# Carpeta local (gitignored) donde están los .sav descomprimidos.
DIR_LOCAL = Path.cwd().parent / "data" / "enaho" / str(ANIO)
DIR_LOCAL.mkdir(parents=True, exist_ok=True)

# Prefijo de las claves en R2 (fuente de verdad).
PREFIJO_R2 = f"enaho/{ANIO}/"

# Nombres de archivo tal como vienen en el .zip del INEI. VERIFICAR contra lo
# que realmente descargaste: el patrón es `Enaho01-<anio>-601.sav` y
# `sumaria-<anio>.sav`, pero el INEI a veces cambia mayúsculas/guiones.
ARCHIVO_601 = f"Enaho01-{ANIO}-601.sav"
ARCHIVO_SUMARIA = f"sumaria-{ANIO}.sav"

print(f"ANIO={ANIO}")
print(f"Local : {DIR_LOCAL}")
print(f"R2    : {PREFIJO_R2}")
print("Archivos presentes en local:", sorted(p.name for p in DIR_LOCAL.glob('*.sav')))

### Descarga desde R2 si no está en local

Reutiliza el cliente boto3 ya configurado en `observatorio.carga.r2_a_supabase` (mismo patrón que el resto del proyecto — no duplicamos credenciales). Si el `.sav` ya está en `data/enaho/<ANIO>/`, no hace nada.

In [ ]:
import os

from observatorio.carga.r2_a_supabase import cliente_r2


def asegurar_local(nombre_archivo: str) -> Path:
    """Devuelve la ruta local del .sav; si no existe, lo baja de R2."""
    destino = DIR_LOCAL / nombre_archivo
    if destino.exists():
        print(f"✅ Ya está en local: {destino.name}")
        return destino

    bucket = os.environ["R2_BUCKET"]
    clave = PREFIJO_R2 + nombre_archivo
    print(f"⬇️  Bajando de R2: {clave}")
    cliente_r2().download_file(bucket, clave, str(destino))
    print(f"✅ Descargado: {destino}")
    return destino


# Si todavía no subiste nada a R2, descarga los .sav manualmente del INEI a
# DIR_LOCAL y esta celda los encontrará sin tocar R2.
ruta_601 = asegurar_local(ARCHIVO_601)
ruta_sumaria = asegurar_local(ARCHIVO_SUMARIA)

## 1. Módulo 601 — Gastos en Alimentos y Bebidas

Es el módulo central para la canasta: una fila por (hogar × producto consumido), con el código del producto, el gasto y la cantidad. Cargamos **datos + metadatos** (las etiquetas son las que traducen los códigos numéricos).

In [ ]:
df601, meta601 = pyreadstat.read_sav(str(ruta_601))

print(f"Filas: {len(df601):,}  ·  Columnas: {df601.shape[1]}")
df601.head()

### 1.1 Diccionario de columnas

`meta.column_names_to_labels` es el diccionario oficial: nombre técnico → descripción legible. Lo dejamos a la vista para identificar las columnas que importan (código de producto, gasto, cantidad, geografía, factor de expansión).

In [ ]:
diccionario_601 = pd.DataFrame(
    meta601.column_names_to_labels.items(),
    columns=["columna", "etiqueta"],
)
pd.set_option("display.max_rows", None)
diccionario_601

In [ ]:
# Columnas candidatas que esperamos usar aguas abajo. VERIFICAR los nombres
# exactos contra el diccionario de arriba (varían algo entre años):
#   - Identificadores de hogar : CONGLOME, VIVIENDA, HOGAR
#   - Geografía                : UBIGEO (6 díg.; 2 primeros = departamento), DOMINIO
#   - Producto                 : P601A  (código del producto/ítem de la canasta)
#   - Gasto / cantidad         : columnas I601* / D601* / G601* (gasto en soles)
#   - Factor de expansión      : FACTOR07 o FACTORA07 (factor anual)
candidatas = ["CONGLOME", "VIVIENDA", "HOGAR", "UBIGEO", "DOMINIO",
              "P601A", "FACTOR07", "FACTORA07"]
presentes = [c for c in candidatas if c in df601.columns]
ausentes = [c for c in candidatas if c not in df601.columns]
print("Presentes:", presentes)
print("Ausentes (buscar equivalente en el diccionario):", ausentes)

### 1.2 Etiquetas de valor (códigos de producto)

`meta.variable_value_labels` mapea cada código numérico a su nombre. La columna de producto (`P601A`) trae aquí el catálogo completo de alimentos: es donde localizaremos papa, limón, pollo, cebolla, huevo y tomate.

In [ ]:
# Ajustar COL_PRODUCTO al nombre real confirmado arriba.
COL_PRODUCTO = "P601A"

etiquetas_producto = meta601.variable_value_labels.get(COL_PRODUCTO, {})
print(f"{COL_PRODUCTO}: {len(etiquetas_producto)} códigos de producto")

catalogo = pd.DataFrame(
    etiquetas_producto.items(), columns=["codigo", "producto"]
).sort_values("codigo")
catalogo.head(30)

### 1.3 Mapeo de los 6 productos del MVP

Buscamos por nombre en el catálogo para fijar los códigos exactos que usaremos en #17/#19. Un mismo alimento puede tener varias presentaciones (p. ej. *papa blanca*, *papa amarilla*): hay que decidir cuáles agrupar.

In [ ]:
objetivos = ["papa", "limon", "limón", "pollo", "cebolla", "huevo", "tomate"]
patron = "|".join(objetivos)

coincidencias = catalogo[
    catalogo["producto"].str.normalize("NFKD")
    .str.encode("ascii", "ignore").str.decode("ascii")
    .str.contains(patron, case=False, na=False)
]
coincidencias

## 2. Sumaria — gasto agregado por hogar

Una fila por hogar, con totales de gasto/ingreso ya calculados por el INEI. Sirve para normalizar (peso del gasto en alimentos sobre el total) y validar agregaciones.

In [ ]:
df_sum, meta_sum = pyreadstat.read_sav(str(ruta_sumaria))
print(f"Filas (hogares): {len(df_sum):,}  ·  Columnas: {df_sum.shape[1]}")
pd.DataFrame(meta_sum.column_names_to_labels.items(), columns=["columna", "etiqueta"]).head(40)

## 3. Factor de expansión

**Crítico.** Cada hogar de la muestra representa a muchos hogares de la población. Para obtener cifras representativas hay que ponderar por el **factor de expansión** (`FACTOR07` / `FACTORA07`), nunca promediar en crudo. Demostramos la agregación correcta por departamento (2 primeros dígitos del `UBIGEO`).

In [ ]:
# Ajustar a los nombres reales confirmados en 1.1.
COL_FACTOR = "FACTORA07" if "FACTORA07" in df601.columns else "FACTOR07"
COL_UBIGEO = "UBIGEO"

df = df601.copy()
df["departamento"] = df[COL_UBIGEO].astype(str).str.zfill(6).str[:2]

# Nº de hogares (muestra) vs. población representada (ponderada) por departamento.
resumen_dept = (
    df.groupby("departamento")
    .agg(
        filas_muestra=(COL_FACTOR, "size"),
        poblacion_estimada=(COL_FACTOR, "sum"),
    )
    .sort_index()
)
resumen_dept

## 4. Subir los crudos a R2 (one-shot)

Ejecutar **una sola vez** tras descargar manualmente del INEI, para dejar los `.sav` como fuente de verdad reproducible. La ENAHO de un año es inmutable, así que no hay re-subidas. Cambiar `SUBIR = True` para activarlo.

In [ ]:
SUBIR = False

if SUBIR:
    s3 = cliente_r2()
    bucket = os.environ["R2_BUCKET"]
    for sav in sorted(DIR_LOCAL.glob("*.sav")):
        clave = PREFIJO_R2 + sav.name
        print(f"⬆️  Subiendo {sav.name} → {clave}")
        s3.upload_file(str(sav), bucket, clave)
    print("✅ Subida completa.")
else:
    print("SUBIR=False — no se subió nada. Cambiar a True para el one-shot inicial.")

## 5. Hallazgos → `docs/enaho.md`

Destilar aquí las conclusiones de la exploración y trasladarlas a [`docs/enaho.md`](../docs/enaho.md), que es lo que alimenta el diseño de la tabla (#17):

- Año usado y archivos (`.sav`) cargados.
- Nombres reales confirmados de: columna de producto, gasto, cantidad, geografía, factor de expansión.
- Códigos ENAHO definitivos de los 6 productos del MVP (y qué presentaciones se agrupan).
- Granularidad de la agregación por departamento y uso del factor de expansión.
- Dudas / decisiones pendientes para #17.